# 02 - AutoML with AutoGluon

AutoGluon trains several model families (LightGBM, CatBoost, XGBoost, random forests, small neural networks), tunes them and stacks the best ones. Here you give it the same data as notebook 01 and see whether AutoML beats a 10-second scikit-learn baseline.

Two things to keep in mind:

* `time_limit` is the **total** training budget in seconds for all models together, not per model.
* AutoGluon needs a validation set. By default it splits training *rows* at random, which would put players of the same match on both sides. We pass our own validation set, split by match.

In [ ]:
import os, sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make the `lolml` helpers importable (notebooks run from notebooks/)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from lolml import data, features as F, paths
paths.ensure_dirs()
SMOKE = bool(os.environ.get("LOLML_SMOKE"))   # LOLML_SMOKE=1 uses the small committed sample and short training budgets
SEED = 42
def save_fig(name):
    plt.tight_layout(); plt.savefig(paths.OUT / name, dpi=110); plt.show()

from autogluon.tabular import TabularPredictor
import autogluon.tabular, sklearn
print("autogluon", autogluon.tabular.__version__, "| scikit-learn", sklearn.__version__)
TIME_LIMIT = 30 if SMOKE else 240    # seconds, whole run

## 1. Post-game model, split by match

Train / validation / test are three groups of *matches*: 64 % / 16 % / 20 %.

In [ ]:
perf = data.load("performance")
perf["win"] = perf["win"].astype(int)
cols = F.post_game_numeric(perf)
tr_idx, te_idx = F.group_split(perf, 0.2, SEED)
trainval, test = perf.iloc[tr_idx], perf.iloc[te_idx]
t_idx, v_idx = F.group_split(trainval, 0.2, SEED + 1)
train, val = trainval.iloc[t_idx], trainval.iloc[v_idx]
for n, d in (("train", train), ("validation", val), ("test", test)):
    print(f"{n:<11}{len(d):>8,} rows {d['match_id'].nunique():>7,} matches")
assert not set(train.match_id) & set(val.match_id) and not set(trainval.match_id) & set(test.match_id)

In [ ]:
label = "win"
predictor = TabularPredictor(label=label, eval_metric="accuracy", path=str(paths.MODELS / "ag_postgame"), verbosity=2).fit(
    train[cols + [label]], tuning_data=val[cols + [label]], time_limit=TIME_LIMIT, presets="medium_quality")

In [ ]:
lb = predictor.leaderboard(test[cols + [label]], silent=True)
lb[["model", "score_test", "score_val", "fit_time", "pred_time_test"]].round(4)

`score_test` is the accuracy on matches the models never saw. `fit_time` is in seconds. Compare with the scikit-learn baseline trained on exactly the same split.

In [ ]:
import time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
t0 = time.time()
hgb = HistGradientBoostingClassifier(max_iter=100 if SMOKE else 300, random_state=SEED).fit(trainval[cols], trainval[label])
sk_time = time.time() - t0
sk_acc = accuracy_score(test[label], hgb.predict(test[cols]))
ag_proba = predictor.predict_proba(test[cols])[1]
ag_acc = accuracy_score(test[label], predictor.predict(test[cols]))
cmp = pd.DataFrame({"accuracy": [sk_acc, ag_acc], "AUC": [roc_auc_score(test[label], hgb.predict_proba(test[cols])[:, 1]), roc_auc_score(test[label], ag_proba)],
                    "training seconds": [sk_time, TIME_LIMIT]}, index=["scikit-learn HistGradientBoosting", f"AutoGluon ({TIME_LIMIT}s budget)"])
print(cmp.round(4).to_string())
cmp.to_json(paths.OUT / "autogluon_vs_sklearn.json")

AutoML is not magic: on a table like this it lands within a fraction of a percentage point of one well-configured gradient boosting model. Where it earns its keep is **saving your time**: no model choice, no tuning, stacking for free, and a leaderboard to inspect.

## 2. What did it learn?

`feature_importance` shuffles each column on a test subset (permutation importance, same idea as scikit-learn) and reports the drop in accuracy.

In [ ]:
sub = test.sample(min(len(test), 3000), random_state=SEED)
fi = predictor.feature_importance(sub[cols + [label]], num_shuffle_sets=3, time_limit=240, silent=True)
fi.head(12)[["importance", "stddev", "p_value"]].round(4)

In [ ]:
fi.head(12)["importance"][::-1].plot.barh(figsize=(6, 4), title="AutoGluon permutation importance (accuracy drop)")
save_fig("autogluon_importance.png")

## 3. Per-minute model with AutoGluon

The same call on the by-minute data. Only the stats the Live Client API can supply (`BYMIN_LIVE_SAFE`), split by match.

In [ ]:
bym = data.load("by_minute")
bc = F.BYMIN_LIVE_SAFE
a_idx, b_idx = F.group_split(bym, 0.2, SEED)
bt, bte = bym.iloc[a_idx], bym.iloc[b_idx]
x_idx, y_idx = F.group_split(bt, 0.2, SEED + 1)
tr_b, va_b = bt.iloc[x_idx], bt.iloc[y_idx]
bpred = TabularPredictor(label="winner", eval_metric="accuracy", path=str(paths.MODELS / "ag_by_minute"), verbosity=1).fit(
    tr_b[bc + ["winner"]], tuning_data=va_b[bc + ["winner"]], time_limit=TIME_LIMIT, presets="medium_quality")
bhgb = HistGradientBoostingClassifier(max_iter=100 if SMOKE else 300, random_state=SEED).fit(tr_b[bc], tr_b["winner"])
b_cmp = pd.DataFrame({"accuracy": [accuracy_score(bte["winner"], bhgb.predict(bte[bc])), accuracy_score(bte["winner"], bpred.predict(bte[bc]))]},
                     index=["scikit-learn HistGradientBoosting", f"AutoGluon ({TIME_LIMIT}s)"])
print(b_cmp.round(4)); b_cmp.to_json(paths.OUT / "autogluon_by_minute.json")

## 4. Keep the model small and portable

An AutoGluon predictor is a folder of models. It can only be loaded by the **same AutoGluon version** that trained it, so record the version next to the model. `clone_for_deployment` keeps only what prediction needs.

In [ ]:
import shutil
def size_mb(p): return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e6
print("model names   :", predictor.model_names()[:6], "...")
print("best model    :", predictor.model_best)
print("trained with  : AutoGluon", (Path(predictor.path) / "version.txt").read_text().strip())
print(f"folder size   : {size_mb(predictor.path):.0f} MB")
slim = predictor.clone_for_deployment(str(paths.MODELS / "ag_postgame_deploy"), dirs_exist_ok=True)
print(f"deploy clone  : {size_mb(slim):.0f} MB")